# Query Rewriting (Enterprise AI Pattern)

Query Rewriting is one of the **most widely used Advanced RAG techniques** in production. It improves retrieval quality by **rewriting the user's question into a clearer, more searchable query** before sending it to the retriever.

Interviewers frequently ask:

- What is Query Rewriting?
- Why do we rewrite queries?
- Query Rewriting vs Multi-Query Retrieval?
- How is Query Rewriting implemented?
- When should it be used?

---

# 1. What is Query Rewriting?

## Definition

**Query Rewriting** is the process of using an LLM to transform a user's original question into a clearer, more specific, and retrieval-friendly query before searching the vector database.

The rewritten query should preserve the original intent while improving retrieval.

---

## Interview Answer

> Query Rewriting is an Advanced RAG technique where an LLM rewrites the user's natural language query into a clearer and retrieval-optimized query. This improves retrieval accuracy by expanding abbreviations, resolving ambiguity, correcting grammar, and adding missing context.

---

# 2. Why Do We Need Query Rewriting?

Users often ask vague or incomplete questions.

Example

```text id="a6hwb1"
PTO Rules?
```

The document contains

```text id="1ntx1x"
Annual Leave Policy
```

Traditional RAG

```text id="d4p5hn"
PTO Rules

↓

Retriever

↓

Poor Match
```

---

Query Rewriting

```text id="6wrqkq"
PTO Rules

↓

Paid Time Off Policy

↓

Annual Leave Policy

↓

Retriever
```

Now retrieval succeeds.

---

# 3. Problems Solved

Query Rewriting helps with:

✅ Abbreviations

```text id="7r7kry"
PTO

↓

Paid Time Off
```

---

✅ Synonyms

```text id="x3z4p2"
Vacation

↓

Annual Leave
```

---

✅ Grammar

```text id="ewr4m6"
leave policy india 2025

↓

What is the leave policy for employees in India for 2025?
```

---

✅ Context

```text id="dgyjlwm"
Show policy

↓

Show HR Leave Policy
```

---

# 4. Architecture

```text id="tpjlwm"
                    User Question
                           │
                           ▼
                  AWS Bedrock /
                 Azure OpenAI
                 (Rewrite Query)
                           │
                           ▼
                  Improved Query
                           │
                           ▼
              OpenSearch / Qdrant /
              Azure AI Search
                           ▼
                    Top-K Chunks
                           ▼
               AWS Bedrock /
               Azure OpenAI
                           ▼
                      Response
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Vector DB | OpenSearch / Qdrant | Azure AI Search |
| Storage | S3 | Blob Storage |
| Backend | ECS/EKS | Container Apps/AKS |

---

# 5. Example

Document

```text id="6g3msd"
Annual Leave Policy
```

---

User asks

```text id="c0q33l"
PTO Rules
```

---

LLM rewrites

```text id="ix8j5z"
What is the company's annual leave policy?
```

Retriever searches using the rewritten query.

---

Another Example

User

```text id="v7d88s"
WFH
```

LLM rewrites

```text id="j2v3lf"
Work From Home Policy
```

---

Healthcare Example

User

```text id="hzt0wq"
High sugar medicine
```

LLM rewrites

```text id="8n5jz7"
Medication for Type 2 Diabetes Mellitus
```

---

# 6. Flow

```text id="djlwm2"
Question

↓

Rewrite Query

↓

Embedding

↓

Retriever

↓

Top-K

↓

LLM

↓

Answer
```

---

# 7. LangChain Example

```python
# ==========================================================
# STEP 1 : Create Bedrock LLM
#
# Purpose:
# The LLM rewrites the user's query.
# ==========================================================

from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1"
)


# ==========================================================
# STEP 2 : Create Rewrite Prompt
#
# Purpose:
# Generate a retrieval-friendly version
# of the user's query.
# ==========================================================

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Rewrite the user's question for document retrieval.

Rules:
- Preserve the original intent.
- Expand abbreviations.
- Resolve ambiguity.
- Return only the rewritten query.
"""
    ),
    ("human", "{question}")
])

rewrite_chain = rewrite_prompt | llm


# ==========================================================
# STEP 3 : Rewrite Query
# ==========================================================

question = "PTO Rules"

rewritten_query = rewrite_chain.invoke(
    {"question": question}
).content

print("Rewritten Query:")
print(rewritten_query)


# ==========================================================
# STEP 4 : Retrieve Documents
#
# Purpose:
# Search the vector database using
# the rewritten query.
# ==========================================================

docs = retriever.invoke(rewritten_query)


# ==========================================================
# STEP 5 : Display Results
# ==========================================================

for doc in docs:
    print(doc.page_content)
```

---

# 8. Production Architecture

```text id="4jlwmq"
User

↓

FastAPI

↓

JWT

↓

LangGraph

↓

Query Rewriter

↓

Hybrid Search

↓

Reranker

↓

Context Compression

↓

Bedrock

↓

Redis

↓

Response
```

---

# 9. Advantages

✅ Better retrieval accuracy

✅ Handles abbreviations

✅ Resolves ambiguous questions

✅ Improves semantic search

✅ Low implementation complexity

---

# 10. Disadvantages

❌ Extra LLM call

❌ Slightly higher latency

❌ Additional token cost

❌ Incorrect rewrites can reduce retrieval quality

---

# 11. Best Practices

✅ Keep the rewritten query concise.

✅ Preserve the original user intent.

✅ Expand abbreviations only when appropriate.

✅ Use together with Hybrid Search.

✅ Evaluate rewritten queries during testing.

---

# 12. Common Mistakes

❌ Changing the meaning of the user's question.

❌ Over-expanding the query with unnecessary information.

❌ Using Query Rewriting for every trivial greeting.

❌ Ignoring domain-specific terminology.

---

# 13. Query Rewriting vs Multi-Query Retrieval

| Query Rewriting | Multi-Query Retrieval |
|-----------------|-----------------------|
| Generates one improved query | Generates multiple query variations |
| One retrieval | Multiple retrievals |
| Lower cost | Higher cost |
| Lower latency | Higher latency |
| Improves precision | Improves recall |

---

Example

Original

```text id="ejlwm7"
PTO Policy
```

---

Query Rewriting

```text id="jlwm88"
Annual Leave Policy
```

↓

One search

---

Multi-Query

```text id="jlwm89"
PTO Policy

Paid Time Off Policy

Annual Leave Policy

Vacation Policy
```

↓

Four searches

---

# 14. Query Rewriting vs Self-Query Retriever

| Query Rewriting | Self-Query Retriever |
|-----------------|----------------------|
| Improves search text | Generates search text + metadata filters |
| No metadata | Uses metadata |
| Simpler | More advanced |
| One search | One filtered search |

---

# 15. Real Enterprise Example

## HR Assistant

User

```text id="jlwm90"
PTO Rules
```

↓

Rewritten

```text id="jlwm91"
Annual Leave Policy
```

↓

Retriever

↓

Correct HR policy

---

## Healthcare Assistant

User

```text id="jlwm92"
High sugar medicine
```

↓

Rewritten

```text id="jlwm93"
Treatment options for Type 2 Diabetes Mellitus
```

↓

Retriever

↓

Clinical guideline

---

# 16. Common Interview Questions

### Q1. Why Query Rewriting?

To improve retrieval by converting vague or ambiguous user questions into retrieval-friendly queries.

---

### Q2. Does Query Rewriting improve recall or precision?

Primarily **precision**, because it produces a better single query.

---

### Q3. Can Query Rewriting be combined with Hybrid Search?

Yes.

A common production pipeline is:

```text id="jlwm94"
Rewrite Query

↓

Hybrid Search

↓

Reranker

↓

LLM
```

---

### Q4. Can Query Rewriting replace Multi-Query Retrieval?

No.

Query Rewriting creates **one** optimized query, whereas Multi-Query Retrieval creates **multiple** query variations to maximize recall.

---

### Q5. Where is Query Rewriting useful?

- HR Assistants
- Banking AI
- Healthcare Assistants
- Legal AI
- Customer Support
- Enterprise Search

---

# 17. Production Enterprise Pipeline

```text id="jlwm95"
PDF

↓

Semantic Chunking

↓

Parent-Child Retrieval

↓

Query Rewriting

↓

Hybrid Search

↓

Self-Query Retriever (Metadata)

↓

Reranker

↓

Context Compression

↓

Bedrock / Azure OpenAI

↓

Response
```

---

# 18. Comparison

| Technique | Main Goal |
|-----------|-----------|
| Query Rewriting | Improve one query |
| Multi-Query Retrieval | Improve recall with multiple queries |
| Self-Query Retriever | Add metadata filtering |
| Hybrid Search | Combine keyword + semantic search |
| Parent-Child Retrieval | Preserve context |
| Semantic Chunking | Create meaningful chunks |

---

# 19. EPAM Senior Answer (3 Minutes)

> "Query Rewriting is an Advanced RAG technique in which an LLM rewrites the user's original question into a clearer, retrieval-optimized query while preserving its intent. This is particularly useful for resolving abbreviations, synonyms, spelling mistakes, and ambiguous wording. For example, a query such as 'PTO Rules' can be rewritten as 'Annual Leave Policy' before retrieval. The rewritten query is then used to search Amazon OpenSearch, Qdrant, or Azure AI Search, and the retrieved documents are passed to Amazon Bedrock or Azure OpenAI for response generation. In production, I typically place Query Rewriting before Hybrid Search and reranking, and I combine it with Semantic Chunking, Parent-Child Retrieval, and metadata filtering to maximize retrieval quality while keeping latency under control. Query Rewriting improves retrieval precision with only one additional LLM call, making it a common optimization in enterprise RAG systems."